In [8]:
import pandas as pd
import sys
import os

sys.path.append("src") 

from data_preprocess import DatasetPreprocessor, DatasetExplorer
from quality_checks import ValidationCheck, BasicCheck, AdvancedCheck, MachineLearningCheck
from llm_assistance import GPTInsightGenerator
from constants import DATA_FILE_PATHS, CLEANED_DATASET_FILE_PATH

## Data Understanding and Preparation

1. Load and pre-clean the data
2. Explore the data
3. Identify top 10 growing markets

In [9]:
data_preprocessor = DatasetPreprocessor(DATA_FILE_PATHS)
data_preprocessor.preclean_and_save_dataset(output_path=CLEANED_DATASET_FILE_PATH)

Preprocessing dataset...
Loading file: input\all_markets_202101_202106_481292_20240618152800.xlsx
Loading file: input\all_markets_202107_202112_481292_20240618152800.xlsx
Loading file: input\all_markets_202201_202206_481292_20240618152800.xlsx
Loading file: input\all_markets_202207_202212_481292_20240618152800.xlsx
Loading file: input\all_markets_202301_202306_481292_20240607130011.xlsx
Loading file: input\all_markets_202307_202312_481292_20240607130013.xlsx
Data loaded successfully, shape: (626652, 14)
Loading file: input\ISO10383_MIC.xlsx


c:\Users\lxh\Anaconda3\envs\abn-case\Lib\site-packages\openpyxl\reader\workbook.py:118: UserWarning: Print area cannot be set to Defined name: ISO10383_MIC!$A:$Q.
  warn(f"Print area cannot be set to Defined name: {defn.value}.")


Loading file: input\ISO4217_Currency_Code.xls
Loading file: input\VIX_History.csv
Dataset loaded and merged successfully, shape: (626652, 24)
Duplicates removed, new shape: (626632, 24)
Column Month could not be converted to numeric, keeping as is.
Column Region could not be converted to numeric, keeping as is.
Column Indicator Name could not be converted to numeric, keeping as is.
Column ExchangeName could not be converted to numeric, keeping as is.
Column CurrencyName could not be converted to numeric, keeping as is.
Column DataType could not be converted to numeric, keeping as is.
Column AggregationType could not be converted to numeric, keeping as is.
Column MARKET NAME-INSTITUTION DESCRIPTION could not be converted to numeric, keeping as is.
Column MIC could not be converted to numeric, keeping as is.
Column OPERATING MIC could not be converted to numeric, keeping as is.
Column Currency could not be converted to numeric, keeping as is.
Column currency_code could not be converted t

c:\Users\lxh\Desktop\ABN-interview\src\data_preprocess\preclean_and_explore.py:234: FutureWarning: The default fill_method='pad' in Series.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  .transform(lambda x: x.pct_change() * 100)


MTM and YTY changes calculated, new shape: (626630, 26)
Columns renamed and selected, new shape: (626630, 22)


In [3]:
df = pd.read_csv(CLEANED_DATASET_FILE_PATH)
data_explorer = DatasetExplorer(df)
data_explorer.explore()

Exploring dataset...
Shape of the DataFrame: (626630, 22)
Columns in the DataFrame: ['Region', 'Indicator Name', 'ExchangeName', 'CurrencyName', 'Value', 'Nominal', 'DataType', 'YTD', '% Change (YTD)', '% Change (MTM)', 'MTM_pct', '% Change (YTY)', 'YTY_pct', 'AggregationType', 'MIC', 'OPERATING MIC', 'currency_code', 'OPEN', 'HIGH', 'LOW', 'CLOSE', 'Date']
Data types of columns:
Region              object
Indicator Name      object
ExchangeName        object
CurrencyName        object
Value              float64
Nominal              int64
DataType            object
YTD                float64
% Change (YTD)     float64
% Change (MTM)     float64
MTM_pct            float64
% Change (YTY)     float64
YTY_pct            float64
AggregationType     object
MIC                 object
OPERATING MIC       object
currency_code       object
OPEN               float64
HIGH               float64
LOW                float64
CLOSE              float64
Date                object
dtype: object
Basic sta

In [4]:
data_explorer.identify_top_n_growing_market_MTM_pct_all_indicators()

Identifying top 10 growing markets based on MTM pct including all indicators...
                             ExchangeName   AVG_MTM_pct
73               indonesia stock exchange  6.525804e+10
113               pakistan stock exchange  2.076759e+09
129                 tehran stock exchange  1.983461e+09
68               hochiminh stock exchange  6.777118e+08
2    angolan securities exchange (bodiva)  5.722630e+08
27            bolsa y mercados argentinos  5.131574e+08
75   iran fara bourse securities exchange  4.548138e+08
31                          boursa kuwait  1.892791e+08
20          bolsa de comercio de santiago  1.090423e+08
81                         korea exchange  1.055900e+08


In [5]:
data_explorer.identify_top_n_growing_market_YTY_pct_all_indicators()

Identifying top 10 growing markets based on YTY pct including all indicators...
                           ExchangeName   AVG_YTY_pct
65                           fmdq group  66990.782101
4         astana international exchange    223.052435
117                qatar stock exchange    134.700000
108    national stock exchange of india    114.202379
3           armenia securities exchange     70.668568
94    multi commodity exchange of india     63.990000
37             bulgarian stock exchange     63.663041
138               warsaw stock exchange     52.645165
64   europe - etp market capitalisation     49.435000
0         abu dhabi securities exchange     41.330892


## Data Quality Checks

1. Validation Checks
2. Basic Checks
3. Advanced Checks
4. Machine Learning Model Check

LLM is used to translate technical findings into insights that are understandable by Regulatory Reporting stakeholders.

In [8]:
# Setup the insight generator
insight_generator = GPTInsightGenerator(api_key=os.getenv('OPENAI_API_KEY'))

### Validation Checks

1. MIC Check: The MIC should be a 4-digit alphanumerical code
2. Currency Code Check: The currency code should be in the ISO 4217 Currency Code

In [10]:
val_check = ValidationCheck(df)
val_results =val_check.run_check()
val_issues_summary = val_check.summarize_issues()
val_issues_summary

{'mic_format_check': {'MIC': {'num_issues': '363950 issues found out of 626630 total rows'}},
 'currency_code_format_check': {'currency_code': {'num_issues': '205949 issues found out of 626630 total rows'}}}

In [11]:
val_check.save_issues("output/val_check_issues.txt")

In [20]:
with open("output/val_check_issues.txt", "r") as f:
    val_check_findings = f.read()

val_check_insight = insight_generator.generate_insight(val_check_findings)

print(val_check_insight)

**Summary of Data Quality Findings**

In our recent data quality assessment, we identified significant issues in two key areas of our financial reporting:

1. **Market Identifier Code (MIC) Format Check:**
   - **Issues Found:** 363,950 out of 626,630 rows
   - This indicates that a substantial portion (approximately 58%) of the MIC entries do not conform to the required format. The MIC is essential for identifying and categorizing trading venues, so errors in this field could lead to incorrect reporting on where transactions occur, affecting transparency and compliance with regulatory standards.

2. **Currency Code Format Check:**
   - **Issues Found:** 205,949 out of 626,630 rows
   - Similar to the MIC, about 33% of currency code entries have discrepancies. The currency code is critical for understanding the type of currency involved in transactions and ensuring accurate financial reporting. Inaccurate currency codes can result in misinterpretations of financial obligations and asse

In [21]:
val_check.export_issues_to_xlsx("output/val_check_issues.xlsx")

### Basic Checks

1. Duplicate Check
2. Missing Values Check
3. Negative Values Check
4. String Length Check
5. Data Types Check

In [13]:
basic_check = BasicCheck(df)
basic_results = basic_check.run_check()
basic_issues_summary = basic_check.summarize_issues()
basic_issues_summary

{'missing_values': {'Value': {'num_issues': '390490 issues found out of 626630 total rows'},
  'YTD': {'num_issues': '460192 issues found out of 626630 total rows'},
  '% Change (YTD)': {'num_issues': '626630 issues found out of 626630 total rows'},
  '% Change (MTM)': {'num_issues': '396152 issues found out of 626630 total rows'},
  'MTM_pct': {'num_issues': '99944 issues found out of 626630 total rows'},
  '% Change (YTY)': {'num_issues': '626630 issues found out of 626630 total rows'},
  'YTY_pct': {'num_issues': '243880 issues found out of 626630 total rows'},
  'MIC': {'num_issues': '363950 issues found out of 626630 total rows'},
  'OPERATING MIC': {'num_issues': '363950 issues found out of 626630 total rows'},
  'currency_code': {'num_issues': '205949 issues found out of 626630 total rows'}}}

In [22]:
basic_check.save_issues("output/basic_check_issues.txt")
with open("output/basic_check_issues.txt", "r") as f:
    basic_check_findings = f.read() 

basic_check_insight = insight_generator.generate_insight(basic_check_findings)
print(basic_check_insight)

**Summary of Data Quality Findings**

We conducted a review of our financial data reporting and identified several significant issues regarding missing values across multiple columns in our dataset of 626,630 rows. The findings are as follows:

1. **Missing Values:**
   - **Value**: 390,490 rows (62% of total)
   - **Year-to-Date (YTD)**: 460,192 rows (73% of total)
   - **% Change (YTD)**: 626,630 rows (100% of total) - every entry is missing.
   - **% Change (Month-to-Month - MTM)**: 396,152 rows (63% of total)
   - **MTM %**: 99,944 rows (16% of total)
   - **% Change (Year-to-Year - YTY)**: 626,630 rows (100% of total) - every entry is missing.
   - **YTY %**: 243,880 rows (39% of total)
   - **MIC**: 363,950 rows (58% of total)
   - **Operating MIC**: 363,950 rows (58% of total)
   - **Currency Code**: 205,949 rows (33% of total)

**Interpretation of Issues:**
- A large percentage of our dataset has missing values, particularly for critical performance indicators like YTD and % ch

In [23]:
basic_check.export_issues_to_xlsx("output/basic_check_issues.xlsx")

MemoryError: 

### Advanced Checks

1. Outlier Dection: Interquartile Range (IQR) method
2. Correlation Analysis
3. Time Series Data Outlier Dection: Rolling Mean and STD Deviation

In [15]:
adv_check = AdvancedCheck(df)
adv_results = adv_check.run_check()
adv_issues_summary = adv_check.summarize_issues()
adv_issues_summary

{'outlier_check_iqr': {'Value': {'abu dhabi securities exchange': {'num_issues': '180 issues found out of 4787 total rows'},
   'amman stock exchange': {'num_issues': '386 issues found out of 4977 total rows'},
   'angolan securities exchange (bodiva)': {'num_issues': '38 issues found out of 1346 total rows'},
   'armenia securities exchange': {'num_issues': '368 issues found out of 4832 total rows'},
   'astana international exchange': {'num_issues': '175 issues found out of 5831 total rows'},
   'asx australian securities exchange': {'num_issues': '649 issues found out of 6223 total rows'},
   'athens stock exchange': {'num_issues': '1150 issues found out of 6606 total rows'},
   'b3 - brasil bolsa balcão': {'num_issues': '454 issues found out of 6074 total rows'},
   'bahrain bourse': {'num_issues': '264 issues found out of 4872 total rows'},
   'baku stock exchange': {'num_issues': '386 issues found out of 4541 total rows'},
   'barbados stock exchange': {'num_issues': '15 issues f

In [ ]:
adv_check.save_issues("output/adv_check_issues.txt")
with open("output/adv_check_issues.txt", "r") as f:
    adv_check_findings = f.read()
adv_check_insight = insight_generator.generate_insight(adv_check_findings)
print(adv_check_insight)

'### Summary of Data Quality Findings for Regulatory Reporting\n\nThis report outlines the data quality issues identified across various financial exchanges, focusing on potential anomalies that could impact compliance and financial reporting processes. The findings are categorized by specific data checks, including outlier detection methods and correlation analysis.\n\n#### Key Findings:\n1. **Outlier Issues by Exchange**: Significant numbers of outlier issues were detected in various exchanges, particularly:\n   - Athens Stock Exchange: 1,150 issues out of 6,606 total entries.\n   - Bolsa de Valores de Colombia: 781 issues out of 6,108 entries.\n   - Bursa Malaysia: 876 issues out of 6,246 entries.\n   - National Stock Exchange of India: 994 issues out of 6,433 entries.\n\n2. **Specific Data Columns**:\n   - **Value**: 1,150 outlier issues were reported in the Athens Stock Exchange, indicating a need for further scrutiny on price anomalies.\n   - **% Change (MTM)**: Notably high issu

In [ ]:
adv_check.export_issues_to_xlsx("output/adv_check_issues.xlsx")

### Machine Learning Model Check

1. Anomaly Detection: Isolation Forest

In [18]:
ml_check = MachineLearningCheck(df)
ml_results = ml_check.run_check()
ml_issues_summary = ml_check.summarize_issues()
ml_issues_summary

{'isolation_forest_anomaly': {'Value': {'abu dhabi securities exchange': {'num_issues': '10 issues found out of 4787 total rows'},
   'amman stock exchange': {'num_issues': '28 issues found out of 4977 total rows'},
   'angolan securities exchange (bodiva)': {'num_issues': '2 issues found out of 1346 total rows'},
   'armenia securities exchange': {'num_issues': '19 issues found out of 4832 total rows'},
   'astana international exchange': {'num_issues': '8 issues found out of 5831 total rows'},
   'asx australian securities exchange': {'num_issues': '32 issues found out of 6223 total rows'},
   'athens stock exchange': {'num_issues': '54 issues found out of 6606 total rows'},
   'b3 - brasil bolsa balcão': {'num_issues': '25 issues found out of 6074 total rows'},
   'bahrain bourse': {'num_issues': '17 issues found out of 4872 total rows'},
   'baku stock exchange': {'num_issues': '18 issues found out of 4541 total rows'},
   'barbados stock exchange': {'num_issues': '1 issues found o

In [19]:
ml_check.save_issues("output/ml_check_issues.txt")
with open("output/ml_check_issues.txt", "r") as f:
    ml_check_findings = f.read()

ml_check_insight = insight_generator.generate_insight(ml_check_findings)
print(ml_check_insight)

### Summary of Data Quality Findings

**Overview of Issues:**
Our analysis of financial data across multiple global exchanges identified a total of 840 issues out of 169,638 records. These issues are flagged anomalies detected through the implementation of an isolation forest model, a machine learning technique used to identify outliers in datasets.

**Detailing the Findings:**
- Many exchanges experienced varying degrees of issues, with the highest incidences found at the **Budapest Stock Exchange** (62 issues) and the **Athens Stock Exchange** (54 issues).
- A number of exchanges, including the **Cayman Island Stock Exchange**, **Barbados Stock Exchange**, and several Euronext locations, reported very few issues, indicating better data quality.
- The distributions of these issues point towards potential inconsistencies, missing values, or data entry errors that could affect market integrity.

**Importance of the Findings:**
Identifying and addressing these data quality issues is crit

In [ ]:
ml_check.export_issues_to_xlsx("output/ml_check_issues.xlsx")